# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# 2. Define Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "fmcg_sales_marketing_profitability_2023_2025.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH

WindowsPath('c:/Users/Anwar/Documents/Work duty/fmcg-business-performance-analytics_new/data/raw/fmcg_sales_marketing_profitability_2023_2025.csv')

# 3. Load Dataset

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

Rows: 18,240
Columns: 27


,Order_ID,Order_Date,Year,Quarter,Month,Month_Name,Region,Country,City,Sales_Person,Customer_Type,Sales_Channel,Promotion_Type,Product_Category,Brand,Product_Name,SKU,Units_Sold,Unit_Price_USD,Discount_Pct,Gross_Sales_USD,Marketing_Spend_USD,COGS_USD,Logistics_Cost_USD,Net_Revenue_USD,Profit_USD,Profit_Margin_Pct
0,FMCG-2025-000001,2025-09-26,2025,Q3,9,September,Oceania,Australia,Perth,Ethan Cole,B2C,Modern Trade,Seasonal Campaign,Personal Care,PureLiva,Shampoo Repair,PCR-PL-101,73,8.47,8.94,618.31,66.05,314.09,43.03,563.03,139.86,24.84
1,FMCG-2024-000002,2024-10-09,2024,Q4,10,October,Asia,India,Mumbai,Meera Nair,B2C,Online,Bundle Offer,Personal Care,BrightSmile,Toothpaste Mint,PCR-BS-301,99,2.89,9.86,286.11,35.26,123.79,23.89,257.90,74.96,29.07
2,FMCG-2024-000003,2024-07-06,2024,Q3,7,July,North America,USA,Los Angeles,Nina Booker,B2B,Distributor,Seasonal Campaign,Personal Care,FreshNest,Body Wash Citrus,PCR-FN-201,361,5.96,15.32,"2,151.56",171.46,"1,011.60",107.02,"1,821.94",531.86,29.19
3,FMCG-2024-000004,2024-05-25,2024,Q2,5,May,Europe,France,Paris,Oliver Kent,B2B,Wholesale,No Promo,Snacks,NutriBite,Protein Bar Peanut,SNK-NB-202,603,3.80,18.00,"2,291.40",118.39,"1,133.66",85.20,"1,878.95",541.70,28.83
4,FMCG-2023-000005,2023-08-10,2023,Q3,8,August,Europe,France,Lyon,Lucas Bennett,B2C,Modern Trade,Loyalty Cashback,Dairy & Breakfast,FarmJoy,Greek Yogurt Plain,DBF-FJ-202,113,3.18,12.46,359.34,34.82,157.70,23.02,314.57,99.03,31.48


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18240 entries, 0 to 18239
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Order_ID             18240 non-null  object 
 1   Order_Date           18240 non-null  object 
 2   Year                 18240 non-null  int64  
 3   Quarter              18240 non-null  object 
 4   Month                18240 non-null  int64  
 5   Month_Name           18240 non-null  object 
 6   Region               18240 non-null  object 
 7   Country              18240 non-null  object 
 8   City                 18240 non-null  object 
 9   Sales_Person         18240 non-null  object 
 10  Customer_Type        18240 non-null  object 
 11  Sales_Channel        18240 non-null  object 
 12  Promotion_Type       18240 non-null  object 
 13  Product_Category     18240 non-null  object 
 14  Brand                18240 non-null  object 
 15  Product_Name         18240 non-null 

In [6]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_count": df.nunique().values,
})

column_summary

,column,dtype,non_null_count,missing_count,missing_pct,unique_count
Order_ID,Order_ID,object,18240,0,0.00,18240
Order_Date,Order_Date,object,18240,0,0.00,1096
Year,Year,int64,18240,0,0.00,3
Quarter,Quarter,object,18240,0,0.00,4
Month,Month,int64,18240,0,0.00,12
Month_Name,Month_Name,object,18240,0,0.00,12
Region,Region,object,18240,0,0.00,5
Country,Country,object,18240,0,0.00,17
City,City,object,18240,0,0.00,48
Sales_Person,Sales_Person,object,18240,0,0.00,36


# 4. EDA

In [11]:
# Create column dictionary

column_dictionary = pd.DataFrame({
    "column": [
        "Order_ID", "Order_Date", "Year", "Quarter", "Month", "Month_Name",
        "Region", "Country", "City",
        "Sales_Person", "Customer_Type", "Sales_Channel", "Promotion_Type",
        "Product_Category", "Brand", "Product_Name", "SKU",
        "Units_Sold", "Unit_Price_USD", "Discount_Pct", "Gross_Sales_USD",
        "Marketing_Spend_USD", "COGS_USD", "Logistics_Cost_USD",
        "Net_Revenue_USD", "Profit_USD", "Profit_Margin_Pct"
    ],
    "business_meaning": [
        "Unique transaction or order identifier",
        "Order transaction date",
        "Transaction year",
        "Transaction quarter",
        "Transaction month number",
        "Transaction month name",
        "Sales region",
        "Country market",
        "City market",
        "Salesperson responsible for the transaction",
        "Customer type such as B2B or B2C",
        "Sales channel such as online, distributor, or modern trade",
        "Promotion campaign type",
        "Product category",
        "Product brand",
        "Product name",
        "Stock keeping unit identifier",
        "Number of units sold",
        "Unit selling price in USD",
        "Discount percentage",
        "Gross sales before cost and profitability adjustments",
        "Marketing spend associated with sales activity",
        "Cost of goods sold",
        "Logistics cost",
        "Net revenue after commercial adjustments",
        "Profit amount",
        "Profit margin percentage"
    ],
    "business_use": [
        "Transaction tracking",
        "Time-series analysis and forecasting",
        "Annual performance analysis",
        "Quarterly business review",
        "Monthly performance tracking",
        "Monthly reporting label",
        "Regional performance analysis",
        "Country-level market analysis",
        "City-level market analysis",
        "Sales performance analysis",
        "Customer segment analysis",
        "Channel performance analysis",
        "Promotion effectiveness analysis",
        "Category portfolio analysis",
        "Brand performance analysis",
        "Product performance analysis",
        "SKU-level portfolio analysis",
        "Volume analysis",
        "Pricing analysis",
        "Discount impact analysis",
        "Topline sales analysis",
        "Marketing efficiency analysis",
        "P&L and margin analysis",
        "Supply chain cost analysis",
        "Revenue performance analysis",
        "Profitability analysis",
        "Margin analysis"
    ]
})

column_dictionary

,column,business_meaning,business_use
0,Order_ID,Unique transaction or order identifier,Transaction tracking
1,Order_Date,Order transaction date,Time-series analysis and forecasting
2,Year,Transaction year,Annual performance analysis
3,Quarter,Transaction quarter,Quarterly business review
4,Month,Transaction month number,Monthly performance tracking
5,Month_Name,Transaction month name,Monthly reporting label
6,Region,Sales region,Regional performance analysis
7,Country,Country market,Country-level market analysis
8,City,City market,City-level market analysis
9,Sales_Person,Salesperson responsible for the transaction,Sales performance analysis


In [9]:
# Missing values and duplicates

missing_summary = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
missing_summary = missing_summary.sort_values("missing_count", ascending=False)

missing_summary

,column,missing_count,missing_pct
0,Order_ID,0,0.00
1,Order_Date,0,0.00
2,Year,0,0.00
3,Quarter,0,0.00
4,Month,0,0.00
5,Month_Name,0,0.00
6,Region,0,0.00
7,Country,0,0.00
8,City,0,0.00
9,Sales_Person,0,0.00


In [12]:
duplicate_count = df.duplicated().sum()
duplicate_order_id_count = df["Order_ID"].duplicated().sum() if "Order_ID" in df.columns else None

print(f"Duplicate full rows: {duplicate_count:,}")
print(f"Duplicate Order_ID values: {duplicate_order_id_count:,}")

Duplicate full rows: 0
Duplicate Order_ID values: 0


In [13]:
# Parse date and validate time range

df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

date_min = df["Order_Date"].min()
date_max = df["Order_Date"].max()

print(f"Date range: {date_min.date()} to {date_max.date()}")
print(f"Invalid dates: {df['Order_Date'].isna().sum():,}")

Date range: 2023-01-01 to 2025-12-31
Invalid dates: 0


In [14]:
df["order_year_from_date"] = df["Order_Date"].dt.year
df["order_month_from_date"] = df["Order_Date"].dt.month

year_mismatch = (df["Year"] != df["order_year_from_date"]).sum()
month_mismatch = (df["Month"] != df["order_month_from_date"]).sum()

print(f"Year mismatch: {year_mismatch:,}")
print(f"Month mismatch: {month_mismatch:,}")

Year mismatch: 0
Month mismatch: 0


In [15]:
# Numerical Summary

numeric_cols = [
    "Units_Sold",
    "Unit_Price_USD",
    "Discount_Pct",
    "Gross_Sales_USD",
    "Marketing_Spend_USD",
    "COGS_USD",
    "Logistics_Cost_USD",
    "Net_Revenue_USD",
    "Profit_USD",
    "Profit_Margin_Pct",
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Units_Sold,"18,240.00",212.91,202.10,5.00,70.00,139.00,288.00,"1,366.00"
Unit_Price_USD,"18,240.00",4.47,1.95,1.45,3.02,3.99,5.44,12.74
Discount_Pct,"18,240.00",12.94,5.74,0.00,8.71,12.69,16.85,28.00
Gross_Sales_USD,"18,240.00",936.43,"1,045.61",13.02,274.53,565.43,"1,213.84","10,634.12"
Marketing_Spend_USD,"18,240.00",85.74,87.55,1.81,33.49,61.39,107.61,"1,767.71"
COGS_USD,"18,240.00",471.02,537.35,6.31,140.51,287.16,599.79,"6,473.33"
Logistics_Cost_USD,"18,240.00",54.03,51.27,1.08,21.37,38.98,69.24,628.86
Net_Revenue_USD,"18,240.00",792.21,861.24,10.54,245.22,493.89,"1,027.36","8,752.94"
Profit_USD,"18,240.00",181.41,239.53,-637.50,32.34,91.89,239.12,"2,723.00"
Profit_Margin_Pct,"18,240.00",19.87,10.57,-45.31,13.72,20.87,27.26,44.49


In [16]:
# Check negative or impossible values

value_checks = []

for col in numeric_cols:
    value_checks.append({
        "column": col,
        "min": df[col].min(),
        "max": df[col].max(),
        "negative_count": (df[col] < 0).sum(),
        "zero_count": (df[col] == 0).sum(),
    })

value_checks_df = pd.DataFrame(value_checks)
value_checks_df

,column,min,max,negative_count,zero_count
0,Units_Sold,5.00,"1,366.00",0,0
1,Unit_Price_USD,1.45,12.74,0,0
2,Discount_Pct,0.00,28.00,0,1
3,Gross_Sales_USD,13.02,"10,634.12",0,0
4,Marketing_Spend_USD,1.81,"1,767.71",0,0
5,COGS_USD,6.31,"6,473.33",0,0
6,Logistics_Cost_USD,1.08,628.86,0,0
7,Net_Revenue_USD,10.54,"8,752.94",0,0
8,Profit_USD,-637.50,"2,723.00",784,2
9,Profit_Margin_Pct,-45.31,44.49,784,3


In [17]:
print("Discount > 100%:", (df["Discount_Pct"] > 100).sum())
print("Profit margin < -100%:", (df["Profit_Margin_Pct"] < -100).sum())
print("Net revenue <= 0:", (df["Net_Revenue_USD"] <= 0).sum())
print("Units sold <= 0:", (df["Units_Sold"] <= 0).sum())

Discount > 100%: 0
Profit margin < -100%: 0
Net revenue <= 0: 0
Units sold <= 0: 0


In [18]:
# Categorical overview

categorical_cols = [
    "Region",
    "Country",
    "City",
    "Sales_Person",
    "Customer_Type",
    "Sales_Channel",
    "Promotion_Type",
    "Product_Category",
    "Brand",
    "Product_Name",
    "SKU",
]

cat_summary = pd.DataFrame({
    "column": categorical_cols,
    "unique_count": [df[col].nunique() for col in categorical_cols],
    "sample_values": [", ".join(df[col].dropna().astype(str).unique()[:5]) for col in categorical_cols],
})

cat_summary

,column,unique_count,sample_values
0,Region,5,"Oceania, Asia, North America, Europe, South Am..."
1,Country,17,"Australia, India, USA, France, Thailand"
2,City,48,"Perth, Mumbai, Los Angeles, Paris, Lyon"
3,Sales_Person,36,"Ethan Cole, Meera Nair, Nina Booker, Oliver Ke..."
4,Customer_Type,2,"B2C, B2B"
5,Sales_Channel,4,"Modern Trade, Online, Distributor, Wholesale"
6,Promotion_Type,7,"Seasonal Campaign, Bundle Offer, No Promo, Loy..."
7,Product_Category,5,"Personal Care, Snacks, Dairy & Breakfast, Hous..."
8,Brand,17,"PureLiva, BrightSmile, FreshNest, NutriBite, F..."
9,Product_Name,30,"Shampoo Repair, Toothpaste Mint, Body Wash Cit..."


In [19]:
# Business KPI quick snapshot

total_gross_sales = df["Gross_Sales_USD"].sum()
total_net_revenue = df["Net_Revenue_USD"].sum()
total_profit = df["Profit_USD"].sum()
total_units = df["Units_Sold"].sum()
total_marketing = df["Marketing_Spend_USD"].sum()
total_cogs = df["COGS_USD"].sum()
total_logistics = df["Logistics_Cost_USD"].sum()

profit_margin = total_profit / total_net_revenue if total_net_revenue != 0 else np.nan
cogs_ratio = total_cogs / total_net_revenue if total_net_revenue != 0 else np.nan
marketing_ratio = total_marketing / total_net_revenue if total_net_revenue != 0 else np.nan
logistics_ratio = total_logistics / total_net_revenue if total_net_revenue != 0 else np.nan

kpi_snapshot = pd.DataFrame({
    "metric": [
        "Gross Sales",
        "Net Revenue",
        "Profit",
        "Units Sold",
        "Marketing Spend",
        "COGS",
        "Logistics Cost",
        "Profit Margin",
        "COGS Ratio",
        "Marketing Spend Ratio",
        "Logistics Cost Ratio",
    ],
    "value": [
        total_gross_sales,
        total_net_revenue,
        total_profit,
        total_units,
        total_marketing,
        total_cogs,
        total_logistics,
        profit_margin,
        cogs_ratio,
        marketing_ratio,
        logistics_ratio,
    ]
})

kpi_snapshot

,metric,value
0,Gross Sales,"17,080,515.05"
1,Net Revenue,"14,449,920.00"
2,Profit,"3,308,965.31"
3,Units Sold,"3,883,474.00"
4,Marketing Spend,"1,563,932.03"
5,COGS,"8,591,482.83"
6,Logistics Cost,"985,539.83"
7,Profit Margin,0.23
8,COGS Ratio,0.59
9,Marketing Spend Ratio,0.11


In [20]:
# Monthly Performance

monthly_perf = (
    df.groupby(["Year", "Month", "Month_Name"], as_index=False)
    .agg(
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
    )
)

monthly_perf["profit_margin"] = monthly_perf["profit"] / monthly_perf["net_revenue"]
monthly_perf["marketing_ratio"] = monthly_perf["marketing_spend"] / monthly_perf["net_revenue"]

monthly_perf.head()

,Year,Month,Month_Name,gross_sales,net_revenue,profit,units_sold,marketing_spend,cogs,logistics_cost,profit_margin,marketing_ratio
0,2023,1,January,"481,511.88","408,301.70","94,361.30",110629,"43,286.62","242,713.37","27,940.41",0.23,0.11
1,2023,2,February,"419,997.02","354,938.26","80,910.28",98261,"39,437.88","210,194.58","24,395.52",0.23,0.11
2,2023,3,March,"438,332.68","371,116.33","84,222.50",105895,"40,046.44","222,070.30","24,777.09",0.23,0.11
3,2023,4,April,"421,101.02","357,347.48","81,665.57",100548,"38,509.72","212,010.72","25,161.47",0.23,0.11
4,2023,5,May,"427,223.81","360,818.12","81,284.58",99014,"38,239.59","216,709.26","24,584.69",0.23,0.11


In [21]:
fig = px.line(
    monthly_perf.sort_values(["Year", "Month"]),
    x="Month_Name",
    y="net_revenue",
    color="Year",
    markers=True,
    title="Monthly Net Revenue by Year",
)

fig.show()

In [22]:
# Cateogory Performance

category_perf = (
    df.groupby("Product_Category", as_index=False)
    .agg(
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
    )
)

category_perf["profit_margin"] = category_perf["profit"] / category_perf["net_revenue"]
category_perf["revenue_share"] = category_perf["net_revenue"] / category_perf["net_revenue"].sum()
category_perf = category_perf.sort_values("net_revenue", ascending=False)

category_perf

,Product_Category,net_revenue,profit,units_sold,marketing_spend,cogs,logistics_cost,profit_margin,revenue_share
0,Beverages,"3,737,688.56","524,201.65",981268,"432,092.21","2,505,214.86","276,179.84",0.14,0.26
2,Household,"3,160,720.02","832,221.58",683008,"320,530.71","1,775,830.31","232,137.42",0.26,0.22
3,Personal Care,"2,765,969.05","776,980.16",686448,"288,589.92","1,523,818.75","176,580.22",0.28,0.19
4,Snacks,"2,516,605.10","555,212.09",912058,"292,218.01","1,510,797.61","158,377.39",0.22,0.17
1,Dairy & Breakfast,"2,268,937.27","620,349.83",620692,"230,501.18","1,275,821.30","142,264.96",0.27,0.16


In [23]:
# Channel performance

channel_perf = (
    df.groupby("Sales_Channel", as_index=False)
    .agg(
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
    )
)

channel_perf["profit_margin"] = channel_perf["profit"] / channel_perf["net_revenue"]
channel_perf["marketing_efficiency"] = channel_perf["profit"] / channel_perf["marketing_spend"].replace(0, np.nan)
channel_perf = channel_perf.sort_values("net_revenue", ascending=False)

channel_perf

,Sales_Channel,net_revenue,profit,units_sold,marketing_spend,profit_margin,marketing_efficiency
3,Wholesale,"5,208,657.92","1,349,279.25",1456048,"419,413.70",0.26,3.22
0,Distributor,"4,451,811.56","1,123,393.98",1227016,"423,220.89",0.25,2.65
1,Modern Trade,"3,371,020.34","660,266.21",860610,"463,912.39",0.20,1.42
2,Online,"1,418,430.18","176,025.87",339800,"257,385.05",0.12,0.68


In [24]:
# Promotion performance

promotion_perf = (
    df.groupby("Promotion_Type", as_index=False)
    .agg(
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        avg_discount=("Discount_Pct", "mean"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
    )
)

promotion_perf["profit_margin"] = promotion_perf["profit"] / promotion_perf["net_revenue"]
promotion_perf["marketing_efficiency"] = promotion_perf["profit"] / promotion_perf["marketing_spend"].replace(0, np.nan)
promotion_perf = promotion_perf.sort_values("net_revenue", ascending=False)

promotion_perf

,Promotion_Type,net_revenue,profit,units_sold,avg_discount,marketing_spend,profit_margin,marketing_efficiency
5,No Promo,"5,587,181.47","1,432,411.66",1438634,8.86,"452,421.75",0.26,3.17
0,Bundle Offer,"2,220,908.58","487,231.45",615463,14.82,"259,855.95",0.22,1.88
6,Seasonal Campaign,"2,216,237.58","477,538.93",602478,13.82,"266,508.37",0.22,1.79
2,Flash Discount,"1,593,444.48","306,545.88",452027,17.72,"236,780.34",0.19,1.29
1,Festival Campaign,"1,183,731.71","218,696.37",329545,16.55,"178,390.14",0.18,1.23
4,Loyalty Cashback,"1,114,225.30","281,314.87",293784,12.23,"91,062.17",0.25,3.09
3,Introductory Offer,"534,190.88","105,226.15",151543,19.46,"78,913.31",0.20,1.33


In [25]:
# Save cleaned dataset

cleaned_preview = df.copy()

cleaned_preview.to_csv(PROCESSED_DATA_DIR / "fmcg_sales_cleaned_preview.csv", index=False)
print("Saved:", PROCESSED_DATA_DIR / "fmcg_sales_cleaned_preview.csv")

Saved: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\processed\fmcg_sales_cleaned_preview.csv


In [26]:
# Initial business observations

observations = []

top_category = category_perf.iloc[0]["Product_Category"]
top_channel = channel_perf.iloc[0]["Sales_Channel"]
top_promotion = promotion_perf.iloc[0]["Promotion_Type"]

observations.append(f"The dataset covers transactions from {date_min.date()} to {date_max.date()}.")
observations.append(f"The top product category by net revenue is {top_category}.")
observations.append(f"The top sales channel by net revenue is {top_channel}.")
observations.append(f"The top promotion type by net revenue is {top_promotion}.")
observations.append(f"Overall profit margin is {profit_margin:.2%}.")
observations.append(f"Marketing spend ratio is {marketing_ratio:.2%} of net revenue.")
observations.append(f"COGS ratio is {cogs_ratio:.2%} of net revenue.")

for obs in observations:
    print("-", obs)

- The dataset covers transactions from 2023-01-01 to 2025-12-31.
- The top product category by net revenue is Beverages.
- The top sales channel by net revenue is Wholesale.
- The top promotion type by net revenue is No Promo.
- Overall profit margin is 22.90%.
- Marketing spend ratio is 10.82% of net revenue.
- COGS ratio is 59.46% of net revenue.
